# FastVision — benchmarks on Colab

Runs the benchmark harness on a **real** multimodal LLM and reports accuracy,
prefill latency, and peak memory across pruning strategies and keep ratios.
Everything runs on the Colab GPU — no local hardware needed.

**Before running:** set `REPO_URL` in the next cell to this repo's GitHub
clone URL (public, or private with a token). A GPU runtime is required
(Runtime → Change runtime type → GPU).

In [ ]:
# 1. Get the code (benchmarks/ is not pip-installable, so we clone) and deps
REPO_URL = "https://github.com/Cekaru/fastvision.git"

!git clone -q $REPO_URL fv_repo
%cd fv_repo
!pip install -q -e ".[bench]" accelerate

In [ ]:
# 2. What to run. LLaVA-1.5-7B in fp16 needs ~16 GB; on the free T4 either
#    switch DTYPE to a 4-bit load or start with Qwen/Qwen2-VL-2B-Instruct.
MODEL_ID    = "llava-hf/llava-1.5-7b-hf"
TASK        = "textvqa"          # textvqa | vqav2 | pope | gqa (textvqa is text-heavy = most pruning-sensitive)
STRATEGIES  = ["divprune", "tome", "fastv", "random"]
KEEP_RATIOS = [1.0, 0.2, 0.1]    # 1.0 is the unpruned baseline (measured once)
LIMIT       = 300                # samples per config; streamed, so only these are downloaded
DTYPE       = "float16"

In [ ]:
# 3. Run the sweep (streams the first LIMIT rows per task — no multi-GB download)
import subprocess, sys

cmd = [
    sys.executable, "-m", "benchmarks.run",
    "--model", MODEL_ID,
    "--task", TASK,
    "--strategies", *STRATEGIES,
    "--keep-ratios", *[str(r) for r in KEEP_RATIOS],
    "--limit", str(LIMIT),
    "--dtype", DTYPE,
    "--output", "benchmark_results",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# 4. Show the real numbers + plots
import os
from IPython.display import Markdown, Image, display

display(Markdown(open("benchmark_results/results.md").read()))
for png in ("accuracy_vs_keep.png", "strategy_comparison.png", "efficiency.png"):
    path = f"benchmark_results/{png}"
    if os.path.exists(path):
        display(Image(path))

## Notes

- `benchmark_results/` holds `results.json`, `results.md`, and the plots.
- Change `MODEL_ID` (e.g. `Qwen/Qwen2-VL-2B-Instruct`) to benchmark another
  family, or raise `LIMIT` for tighter estimates.
- On the free T4, load a 7B model in 4-bit or pick a ~2B model.